# Pikksilm

* Markus Kont
* Stamus Networks
* markus@stamus-networks.com
* github.com/markuskont

## Agenda

* Talk about a tool I built
* Show, not tell
* Pray to demo gods
* Plan A-B-C
* (slides are plan C)

## /whoami

* Background
    * Rooted in Infra and DevOps
    * NATO CCDCOE Tech Researcher and PhD studies
* Currently
    * Stamus Networks since 2020
    * Data analytics, threat hunting, software engineering, devops, etc
    * Going off the reservation
* Still **NOT** a Windows guy
* Nor a red teamer

<img src="me.jpeg" width="300">

## Background

### Locked Shields 2022

* First time attending in blue team
* Role - network detection (suricata, arkime)
* Question in Discord:
    * "anyone knows Kafka streaming?"
    * it was actually not about kafka streaming at all

### Setup - Technical

* Suricata, Arkime
* Sysmon, winlogbeat
* Elastic everything - Stamus, Onprem, Cloud
* Double, triple, quadruple ship all the logs!

### Problem - humans

* Mixed bag of volunteers
* Network hunters do not understand endpoint - and vice versa
* Tooling used by either side totally different
* Both sides needed - much confusion
* Incident trancends technology

### Tech lead be like

<img src="cross-the-streams.png" width="600">

## Grand plan

* Sysmon Event ID 1 - Process Created
    * full command line, PE info, parent process info
* Sysmon EVENT ID 3 - Network Connection
    * basic process info (no parent), 5-tuple, **community ID**
* Correlate using process **Entity ID**
* Push each correlation to redis
    * key is community ID, Arkime WISE configured to look up
* Embed correlations to Suricata EVE using **community ID**

In [ ]:
import json

In [ ]:
with open("./winlog.json", "r") as handle:
    DATA = [json.loads(line) for line in handle]

In [ ]:
from IPython.display import JSON

In [ ]:
JSON(DATA)

### Profit

<img src="arkime.png" width="1000">

### Pikksilm

* https://github.com/markuskont/pikksilm
* Estonian for `looking glass`
    * Too many projects already use that one
    * More direct translation would be `spyglass`
* Written in Golang
* Core work done in 2022/2023
* Updated 2 weeks before the conference (because obviously)

## Tuoni.io

* Some Estonian pentesters decided Cobalt Strike is not good enough
* https://tuoni.io/
* TLS callbacks
* CDN proxy
* https://drive.google.com/file/d/1fH6rVs_lfaUMraykEHTPzBjgkuJq7QYd/view?usp=drive_link

### Vanilla Suricata

<img src="meerkat-sad.png" width="400">

In [ ]:
import pandas as pd

In [ ]:
with open("./suricata-vanilla.json", "r") as handle:
    DATA = [json.loads(line) for line in handle]
DF = pd.json_normalize(DATA)

In [ ]:
DF.columns.values

In [ ]:
(
    DF
    .sort_values(by="timestamp")
    [["community_id", "src_ip", "dest_ip", "dest_port", "app_proto", "alert.signature", "tls.ja4", "tls.version", "flow.age"]]
)

### Suricata on steroids

<img src="meerkat-on-steroids.png" width="500">

In [ ]:
with open("./suricata-enriched.json", "r") as handle:
    DATA = [json.loads(line) for line in handle]
DF = pd.json_normalize(DATA)

In [ ]:
[c for c in DF.columns.values if c.startswith("edr.")]

In [ ]:
(
    DF
    .sort_values(by="timestamp")
    [["community_id", "src_ip", "dest_ip", "dest_port", "tls.ja4", "edr.process.command_line", "edr.process.parent.command_line", "edr.user.name", "edr.process.working_directory", "edr.process.parent.pid"]]
)

In [ ]:
(
    DF
    .groupby("edr.process.name")
    .agg({
        "src_ip": ["unique"],
        "dest_ip": ["unique"],
        "event_type": ["unique"],
        "edr.process.parent.name": ["unique"],
        "tls.ja4": ["unique"],
        "flow.bytes_toserver": ["sum", "mean"],
        "flow.bytes_toclient": ["sum", "mean"],
        "flow.age": ["sum", "mean"],
        "alert.signature": ["unique"]
    })
)

## Limitations

* Prototype for lab work and exercises
* Only two sysmon *opcodes* supported
* Suricata stream must be delayed

### Why not SIEM?

* Why not indeed
* Tool vs tool vs tool
* It's really about **knowledge**
* Reducing friction between hunters
* Even experienced people get tunnel vision
* Lab glue

### Where to from here

* Rethink the concept - enrichment is secondary
* Generic Sysmon Aggregator
    * loaded DLLs, Named Pipes, Created files, Registry modifications, etc
    * all represented in single event
* Scalable Suricata support - separate EVE event_type;

## Thank you!